In [47]:
from __future__ import annotations

import re
import sys
import json
from pathlib import Path
from collections import Counter

ROOT = Path(r"C:\Users\12524\Desktop\Koto")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from bs4 import BeautifulSoup
from docx import Document
from docx.document import Document as _Document
from docx.table import Table, _Cell
from docx.text.paragraph import Paragraph
from docx.oxml.ns import qn

from app.core.file.file_parser import parse_docx

DOC_INVEST = ROOT / "workspace" / "雷鸟创新-投资建议书 (1).docx"
DOC_RESUME = ROOT / "workspace" / "王宇轩-简历（美元).docx"

ALIGN_MAP = {
    "left": "left",
    "center": "center",
    "right": "right",
    "both": "justify",
    "justify": "justify",
    "distribute": "justify",
}

HEADING_MAP = {
    "heading 1": "h1", "heading 2": "h2", "heading 3": "h3",
    "heading 4": "h4", "heading 5": "h5", "heading 6": "h6",
    "heading1": "h1", "heading2": "h2", "heading3": "h3",
    "heading4": "h4", "heading5": "h5", "heading6": "h6",
    "标题 1": "h1", "标题 2": "h2", "标题 3": "h3",
    "标题 4": "h4", "标题 5": "h5", "标题 6": "h6",
    "标题1": "h1", "标题2": "h2", "标题3": "h3",
    "标题4": "h4", "标题5": "h5", "标题6": "h6",
    "一级标题": "h1", "二级标题": "h2", "三级标题": "h3",
    "标题": "h1", "subheading 1": "h2", "subheading 2": "h3",
}


def norm_style_key(val: str | None) -> str:
    if not val:
        return ""
    return str(val).strip().lower().replace(" ", "").replace("_", "")


def is_heading_style_key(val: str | None) -> str | None:
    if not val:
        return None
    raw = str(val).strip().lower()
    if raw in HEADING_MAP:
        return HEADING_MAP[raw]
    key = norm_style_key(val)
    if key.startswith("heading") and len(key) > 7 and key[7:].isdigit():
        level = int(key[7:])
        if 1 <= level <= 6:
            return f"h{level}"
    if key.startswith("标题"):
        digits = "".join(ch for ch in key if ch.isdigit())
        if digits:
            level = int(digits)
            if 1 <= level <= 6:
                return f"h{level}"
    return None


def read_on_off_prop(el) -> bool | None:
    if el is None:
        return None
    val = el.get(qn("w:val"))
    if val is None or str(val).strip() == "":
        return True
    norm = str(val).strip().lower()
    if norm in ("0", "false", "off", "no"):
        return False
    return True


def iter_paragraphs(parent, path="body"):
    from docx.oxml.text.paragraph import CT_P
    from docx.oxml.table import CT_Tbl

    if isinstance(parent, _Document):
        parent_elm = parent.element.body
    elif isinstance(parent, _Cell):
        parent_elm = parent._tc
    else:
        raise TypeError(type(parent))

    para_idx = 0
    table_idx = 0
    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent), f"{path}/p[{para_idx}]"
            para_idx += 1
        elif isinstance(child, CT_Tbl):
            table = Table(child, parent)
            seen_cells: set[int] = set()
            for row_idx, row in enumerate(table.rows):
                for col_idx, cell in enumerate(row.cells):
                    cell_key = id(cell._tc)
                    if cell_key in seen_cells:
                        continue
                    seen_cells.add(cell_key)
                    yield from iter_paragraphs(cell, f"{path}/tbl[{table_idx}]/r[{row_idx}]/c[{col_idx}]")
            table_idx += 1


def clean_text(text: str | None) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def style_chain(para: Paragraph) -> list[dict[str, str]]:
    chain: list[dict[str, str]] = []
    try:
        style = para.style if para.style else None
    except Exception:
        style = None
    visited: set[int] = set()
    while style is not None:
        key = id(getattr(style, "_element", None))
        if key in visited:
            break
        visited.add(key)
        chain.append({
            "name": getattr(style, "name", "") or "",
            "id": getattr(style, "style_id", "") or "",
        })
        try:
            style = style.base_style
        except Exception:
            break
    return chain


def chain_has_heading_style(chain: list[dict[str, str]]) -> bool:
    return any(is_heading_style_key(item["name"]) or is_heading_style_key(item["id"]) for item in chain)


def direct_outline_level(para: Paragraph):
    p_pr = para._element.find(qn("w:pPr"))
    if p_pr is None:
        return None
    outline = p_pr.find(qn("w:outlineLvl"))
    if outline is None:
        return None
    raw = outline.get(qn("w:val"))
    try:
        return int(raw)
    except Exception:
        return raw


def direct_alignment(para: Paragraph):
    p_pr = para._element.find(qn("w:pPr"))
    if p_pr is not None:
        jc = p_pr.find(qn("w:jc"))
        if jc is not None:
            raw = jc.get(qn("w:val")) or jc.get("val")
            if raw:
                return ALIGN_MAP.get(raw, raw)
    try:
        if para.alignment is not None:
            raw = str(para.alignment).split(".")[-1].lower()
            return ALIGN_MAP.get(raw, raw)
    except Exception:
        pass
    return None


def para_rpr(para: Paragraph):
    p_pr = para._element.find(qn("w:pPr"))
    return p_pr.find(qn("w:rPr")) if p_pr is not None else None


def para_bold_state(para: Paragraph):
    r_pr = para_rpr(para)
    if r_pr is None:
        return None
    state = read_on_off_prop(r_pr.find(qn("w:b")))
    if state is None:
        state = read_on_off_prop(r_pr.find(qn("w:bCs")))
    return state


def para_font_size(para: Paragraph):
    r_pr = para_rpr(para)
    if r_pr is None:
        return None
    sz = r_pr.find(qn("w:sz"))
    if sz is None:
        sz = r_pr.find(qn("w:szCs"))
    if sz is None:
        return None
    raw = sz.get(qn("w:val")) or sz.get("val")
    try:
        return round(int(raw) / 2, 1)
    except Exception:
        return raw


def run_bold_summary(para: Paragraph) -> dict[str, int | str]:
    counter = Counter(on=0, off=0, unset=0)
    for run in para.runs:
        r_pr = run._element.find(qn("w:rPr"))
        if r_pr is None:
            counter["unset"] += 1
            continue
        state = read_on_off_prop(r_pr.find(qn("w:b")))
        if state is None:
            state = read_on_off_prop(r_pr.find(qn("w:bCs")))
        if state is True:
            counter["on"] += 1
        elif state is False:
            counter["off"] += 1
        else:
            counter["unset"] += 1
    total = sum(counter.values())
    if total == 0:
        dominant = "no-runs"
    else:
        dominant = max(("on", "off", "unset"), key=lambda key: counter[key])
    return {
        "on": counter["on"],
        "off": counter["off"],
        "unset": counter["unset"],
        "total": total,
        "dominant": dominant,
    }


def has_toc_anchor(p_el) -> bool:
    try:
        for hyperlink in p_el.findall(qn("w:hyperlink")):
            anchor = (hyperlink.get(qn("w:anchor")) or "").lower()
            if anchor.startswith("_toc"):
                return True
    except Exception:
        return False
    return False


def has_toc_field(p_el) -> bool:
    try:
        instr = " ".join((node.text or "") for node in p_el.iter(qn("w:instrText")))
        instr_norm = re.sub(r"\s+", " ", instr).strip().upper()
        if not instr_norm:
            return False
        return (
            " TOC " in f" {instr_norm} "
            or (" PAGEREF " in f" {instr_norm} " and "_TOC" in instr_norm)
        )
    except Exception:
        return False


def p_elem_text_content(p_el) -> str:
    try:
        return clean_text("".join((node.text or "") for node in p_el.iter(qn("w:t"))))
    except Exception:
        return ""


def looks_like_toc_line(p_el) -> bool:
    text = p_elem_text_content(p_el)
    if not text or len(text) > 160:
        return False
    return re.match(r"^.+?\d{1,4}$", text) is not None


def paragraph_looks_like_toc(para: Paragraph, chain: list[dict[str, str]]) -> bool:
    p_el = para._element
    style_candidates = []
    if chain:
        style_candidates.extend([chain[0]["name"], chain[0]["id"]])
    p_pr = p_el.find(qn("w:pPr"))
    if p_pr is not None:
        p_style = p_pr.find(qn("w:pStyle"))
        if p_style is not None:
            style_candidates.append(p_style.get(qn("w:val")) or "")

    has_signal = (
        has_toc_anchor(p_el)
        or has_toc_field(p_el)
        or p_el.find(".//" + qn("w:tab")) is not None
        or looks_like_toc_line(p_el)
    )

    for candidate in style_candidates:
        norm = norm_style_key(candidate)
        if norm and ("toc" in norm or "目录" in norm or "tableofcontents" in norm):
            if has_signal:
                return True

    if p_el.find(".//" + qn("w:tab")) is not None and (
        has_toc_anchor(p_el) or has_toc_field(p_el) or looks_like_toc_line(p_el)
    ):
        return True
    return False


def extract_html_blocks(html: str) -> list[dict[str, str]]:
    soup = BeautifulSoup(html, "html.parser")
    blocks = []
    for tag in soup.find_all(["p", "h1", "h2", "h3", "h4", "h5", "h6"]):
        blocks.append({
            "tag": tag.name,
            "text": clean_text(tag.get_text(" ", strip=True)),
            "style": tag.get("style", "") or "",
            "class": " ".join(tag.get("class", [])),
            "id": tag.get("id", "") or "",
        })
    return blocks


def map_blocks(paragraphs: list[dict], blocks: list[dict]) -> None:
    block_idx = 0
    for para in paragraphs:
        para["html_tag"] = None
        para["html_style"] = ""
        para["html_class"] = ""
        para["html_id"] = ""
        para["in_parse_headings"] = False
        target = clean_text(para["text"])
        if not target:
            continue
        for idx in range(block_idx, len(blocks)):
            block = blocks[idx]
            if clean_text(block["text"]) == target:
                para["html_tag"] = block["tag"]
                para["html_style"] = block["style"]
                para["html_class"] = block["class"]
                para["html_id"] = block["id"]
                block_idx = idx + 1
                break


def analyze_doc(path: Path) -> dict:
    doc = Document(str(path))
    parsed = parse_docx(str(path))
    html = parsed.get("html", "")
    headings = parsed.get("headings", [])
    blocks = extract_html_blocks(html)

    paragraphs = []
    for idx, (para, location) in enumerate(iter_paragraphs(doc)):
        text = clean_text(para.text)
        if not text:
            continue
        chain = style_chain(para)
        direct_style = chain[0] if chain else {"name": "", "id": ""}
        heading_hit = None
        for entry in headings:
            if clean_text(entry.get("text", "")) == text:
                heading_hit = entry
                break
        paragraphs.append({
            "index": idx,
            "location": location,
            "text": text,
            "snippet": text[:120],
            "style_name": direct_style["name"],
            "style_id": direct_style["id"],
            "style_chain": chain,
            "style_chain_has_heading": chain_has_heading_style(chain),
            "outlineLvl": direct_outline_level(para),
            "alignment": direct_alignment(para),
            "ppr_bold": para_bold_state(para),
            "ppr_font_size": para_font_size(para),
            "run_bold": run_bold_summary(para),
            "looks_like_toc": paragraph_looks_like_toc(para, chain),
            "heading_entry": heading_hit,
        })

    map_blocks(paragraphs, blocks)
    heading_texts = {clean_text(item.get("text", "")) for item in headings}
    for para in paragraphs:
        para["in_parse_headings"] = para["text"] in heading_texts
    return {
        "path": str(path),
        "headings": headings,
        "blocks": blocks,
        "paragraphs": paragraphs,
    }


analysis = {
    "investment": analyze_doc(DOC_INVEST),
    "resume": analyze_doc(DOC_RESUME),
}

summary = {
    name: {
        "paragraph_count": len(data["paragraphs"]),
        "heading_count": len(data["headings"]),
        "heading_samples": data["headings"][:12],
    }
    for name, data in analysis.items()
}

print(json.dumps(summary, ensure_ascii=False, indent=2))

{
  "investment": {
    "paragraph_count": 1042,
    "heading_count": 38,
    "heading_samples": [
      {
        "level": 1,
        "text": "执行概要",
        "id": ""
      },
      {
        "level": 2,
        "text": "一、企业简介",
        "id": "_Toc198131814"
      },
      {
        "level": 2,
        "text": "二、投资亮点",
        "id": "_Toc198131815"
      },
      {
        "level": 2,
        "text": "三、投资方案",
        "id": "_Toc198131816"
      },
      {
        "level": 1,
        "text": "第一章 公司基本信息",
        "id": ""
      },
      {
        "level": 2,
        "text": "一、公司简介",
        "id": "_Toc198131818"
      },
      {
        "level": 2,
        "text": "二、工商信息",
        "id": "_Toc198131819"
      },
      {
        "level": 2,
        "text": "三、股权结构",
        "id": "_Toc198131820"
      },
      {
        "level": 2,
        "text": "四、公司历次融资情况",
        "id": "_Toc198131821"
      },
      {
        "level": 2,
        "text": "五、历史沿革及管理架构",
        "id": "_Toc198131

In [3]:
import json

def compact(row):
    return {
        "index": row["index"],
        "location": row["location"],
        "snippet": row["snippet"],
        "style": f"{row['style_name']} / {row['style_id']}",
        "chain_has_heading": row["style_chain_has_heading"],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "ppr_font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "looks_like_toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "html_style": row["html_style"],
        "in_parse_headings": row["in_parse_headings"],
        "heading_entry": row["heading_entry"],
    }

investment = analysis["investment"]["paragraphs"]
rendered_heading_rows = [
    row for row in investment
    if row["html_tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"} or row["in_parse_headings"]
]
outline_only_headings = [
    compact(row) for row in rendered_heading_rows
    if row["outlineLvl"] is not None and not row["style_chain_has_heading"]
]
phrase_hits = [
    compact(row) for row in investment
    if "作为国内AI眼镜领域的链主企业" in row["text"]
]

print("outline_only_headings_count=", len(outline_only_headings))
print(json.dumps(outline_only_headings[:40], ensure_ascii=False, indent=2))
print("phrase_hits=")
print(json.dumps(phrase_hits, ensure_ascii=False, indent=2))

outline_only_headings_count= 31
[
  {
    "index": 43,
    "location": "body/p[36]",
    "snippet": "二、投资亮点",
    "style": "Normal / 1",
    "chain_has_heading": false,
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "ppr_font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "looks_like_toc": false,
    "html_tag": "h2",
    "html_style": "margin-bottom:8.0pt;line-height:1.1583;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "in_parse_headings": true,
    "heading_entry": {
      "level": 2,
      "text": "二、投资亮点",
      "id": "_Toc198131815"
    }
  },
  {
    "index": 60,
    "location": "body/p[47]",
    "snippet": "三、投资方案",
    "style": "List Paragraph / 54",
    "chain_has_heading": false,
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "ppr_font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total":

In [4]:
def tiny(row):
    return {
        "index": row["index"],
        "snippet": row["snippet"],
        "style_name": row["style_name"],
        "style_id": row["style_id"],
        "chain_has_heading": row["style_chain_has_heading"],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "ppr_font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "looks_like_toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "in_parse_headings": row["in_parse_headings"],
    }

investment = analysis["investment"]["paragraphs"]
outline_only = [
    row for row in investment
    if row["html_tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"}
    and row["outlineLvl"] is not None
    and not row["style_chain_has_heading"]
]
print("outline_only rendered headings:", len(outline_only))
print(json.dumps([tiny(row) for row in outline_only[:12]], ensure_ascii=False, indent=2))
print("phrase hit count:", sum("作为国内AI眼镜领域的链主企业" in row["text"] for row in investment))
for row in investment:
    if "作为国内AI眼镜领域的链主企业" in row["text"]:
        print(json.dumps(tiny(row), ensure_ascii=False, indent=2))
        break

outline_only rendered headings: 11
[
  {
    "index": 43,
    "snippet": "二、投资亮点",
    "style_name": "Normal",
    "style_id": "1",
    "chain_has_heading": false,
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "ppr_font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "looks_like_toc": false,
    "html_tag": "h2",
    "in_parse_headings": true
  },
  {
    "index": 60,
    "snippet": "三、投资方案",
    "style_name": "List Paragraph",
    "style_id": "54",
    "chain_has_heading": false,
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "ppr_font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "looks_like_toc": false,
    "html_tag": "h2",
    "in_parse_headings": true
  },
  {
    "index": 81,
    "snippet": "第一章 公司基本信息",
    "style_name": "Normal",
    "style_id": "1",
    "chain_ha

In [26]:
investment = analysis["investment"]["paragraphs"]
outline_only = [
    row for row in investment
    if row["html_tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"}
    and row["outlineLvl"] is not None
    and not row["style_chain_has_heading"]
]
mini = []
for row in outline_only[:3]:
    mini.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "in_headings": row["in_parse_headings"],
    })
phrase = next((row for row in investment if "作为国内AI眼镜领域的链主企业" in row["text"]), None)
phrase_out = None if phrase is None else {
    "index": phrase["index"],
    "snippet": phrase["snippet"],
    "style": [phrase["style_name"], phrase["style_id"]],
    "outlineLvl": phrase["outlineLvl"],
    "alignment": phrase["alignment"],
    "ppr_bold": phrase["ppr_bold"],
    "font_size": phrase["ppr_font_size"],
    "run_bold": phrase["run_bold"],
    "toc": phrase["looks_like_toc"],
    "html_tag": phrase["html_tag"],
    "in_headings": phrase["in_parse_headings"],
}
print(json.dumps({"outline_only_first3": mini, "phrase": phrase_out}, ensure_ascii=False, indent=2))

{
  "outline_only_first3": [
    {
      "index": 43,
      "snippet": "二、投资亮点",
      "style": [
        "Normal",
        "1"
      ],
      "outlineLvl": 1,
      "alignment": null,
      "ppr_bold": true,
      "font_size": 14.0,
      "run_bold": {
        "on": 1,
        "off": 0,
        "unset": 0,
        "total": 1,
        "dominant": "on"
      },
      "toc": false,
      "html_tag": "h2",
      "in_headings": true
    },
    {
      "index": 60,
      "snippet": "三、投资方案",
      "style": [
        "List Paragraph",
        "54"
      ],
      "outlineLvl": 1,
      "alignment": null,
      "ppr_bold": true,
      "font_size": 14.0,
      "run_bold": {
        "on": 1,
        "off": 0,
        "unset": 0,
        "total": 1,
        "dominant": "on"
      },
      "toc": false,
      "html_tag": "h2",
      "in_headings": true
    },
    {
      "index": 81,
      "snippet": "第一章 公司基本信息",
      "style": [
        "Normal",
        "1"
      ],
      "outlineLvl": 0,
     

In [27]:
investment = analysis["investment"]["paragraphs"]
suspicious_headingish = []
for row in investment:
    if row["html_tag"] not in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        continue
    text = row["text"]
    if len(text) > 18 or any(ch in text for ch in "，。；：、“”《》（）"):
        suspicious_headingish.append({
            "index": row["index"],
            "snippet": row["snippet"],
            "style": [row["style_name"], row["style_id"]],
            "outlineLvl": row["outlineLvl"],
            "alignment": row["alignment"],
            "ppr_bold": row["ppr_bold"],
            "font_size": row["ppr_font_size"],
            "run_bold": row["run_bold"],
            "toc": row["looks_like_toc"],
            "html_tag": row["html_tag"],
            "in_headings": row["in_parse_headings"],
        })
print(json.dumps(suspicious_headingish[:12], ensure_ascii=False, indent=2))

[
  {
    "index": 37,
    "snippet": "一、企业简介",
    "style": [
      "标题2",
      "72"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": null,
    "font_size": null,
    "run_bold": {
      "on": 0,
      "off": 0,
      "unset": 1,
      "total": 1,
      "dominant": "unset"
    },
    "toc": false,
    "html_tag": "h2",
    "in_headings": true
  },
  {
    "index": 43,
    "snippet": "二、投资亮点",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "toc": false,
    "html_tag": "h2",
    "in_headings": true
  },
  {
    "index": 60,
    "snippet": "三、投资方案",
    "style": [
      "List Paragraph",
      "54"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
    

In [28]:
investment = analysis["investment"]["paragraphs"]
outline_body = []
for row in investment:
    if row["outlineLvl"] is None:
        continue
    if row["style_chain_has_heading"]:
        continue
    if row["html_tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"}:
        continue
    outline_body.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "in_headings": row["in_parse_headings"],
    })
print("outline-marked body paragraphs:", len(outline_body))
print(json.dumps(outline_body[:12], ensure_ascii=False, indent=2))

outline-marked body paragraphs: 20
[
  {
    "index": 193,
    "snippet": "五、历史沿革及管理架构",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "toc": false,
    "html_tag": null,
    "in_headings": true
  },
  {
    "index": 597,
    "snippet": "六、关联公司",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "toc": false,
    "html_tag": null,
    "in_headings": true
  },
  {
    "index": 622,
    "snippet": "七、公司团队",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
     

In [8]:
investment = analysis["investment"]["paragraphs"]
outline_body = [
    row for row in investment
    if row["outlineLvl"] is not None
    and not row["style_chain_has_heading"]
    and row["html_tag"] not in {"h1", "h2", "h3", "h4", "h5", "h6"}
]
small = []
for row in outline_body[:5]:
    small.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
        "in_headings": row["in_parse_headings"],
    })
print(json.dumps(small, ensure_ascii=False, indent=2))

[
  {
    "index": 177,
    "snippet": "五、历史沿革及管理架构",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": null,
    "in_headings": true
  },
  {
    "index": 545,
    "snippet": "六、关联公司",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": null,
    "in_headings": true
  },
  {
    "index": 568,
    "snippet": "七、公司团队",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
 

In [9]:
investment = analysis["investment"]["paragraphs"]
not_in_headings = []
for row in investment:
    if row["outlineLvl"] is None:
        continue
    if row["style_chain_has_heading"]:
        continue
    if row["in_parse_headings"]:
        continue
    not_in_headings.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
        "in_headings": row["in_parse_headings"],
        "length": len(row["text"]),
    })
print("count", len(not_in_headings))
print(json.dumps(not_in_headings[:12], ensure_ascii=False, indent=2))

count 0
[]


In [29]:
blocks = analysis["investment"]["blocks"]
heading_blocks = [b for b in blocks if b["tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"}]
odd_heading_blocks = [
    {
        "tag": b["tag"],
        "text": b["text"][:140],
        "length": len(b["text"]),
        "style": b["style"],
        "class": b["class"],
        "id": b["id"],
    }
    for b in heading_blocks
    if len(b["text"]) > 18 or any(ch in b["text"] for ch in "，。；：、“”《》（）")
]
print("heading block count", len(heading_blocks))
print(json.dumps(odd_heading_blocks[:20], ensure_ascii=False, indent=2))

heading block count 34
[
  {
    "tag": "h2",
    "text": "一、企业简介",
    "length": 6,
    "style": "margin-bottom:5.0pt;line-height:1.0792;font-size:12.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_Toc198131814"
  },
  {
    "tag": "h1",
    "text": "雷鸟创新技术（ 深圳）有限公司成立于 2021 年，是 TCL 集团孵化的高新技术企业，专注于消费级显示+智能眼镜开发，致力于推动 AI+AR 硬件及场景应用的规模化落地。",
    "length": 86,
    "style": "text-align:center;margin-bottom:8.0pt;line-height:1.1583;font-size:12.0pt;font-family:'STFangsong';font-weight:bold;text-indent:24.0pt",
    "class": "",
    "id": ""
  },
  {
    "tag": "h2",
    "text": "二、投资亮点",
    "length": 6,
    "style": "margin-bottom:5.0pt;line-height:1.0792;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_Toc198131815"
  },
  {
    "tag": "h2",
    "text": "三、投资方案",
    "length": 6,
    "style": "margin-bottom:8.0pt;line-height:1.1583;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_T

In [30]:
blocks = analysis["investment"]["blocks"]
heading_blocks = [b for b in blocks if b["tag"] in {"h1", "h2", "h3", "h4", "h5", "h6"}]
heading_blocks = sorted(heading_blocks, key=lambda b: len(b["text"]), reverse=True)
print(json.dumps([
    {
        "tag": b["tag"],
        "text": b["text"][:140],
        "length": len(b["text"]),
        "style": b["style"],
        "class": b["class"],
        "id": b["id"],
    }
    for b in heading_blocks[:8]
], ensure_ascii=False, indent=2))

[
  {
    "tag": "h1",
    "text": "雷鸟创新技术（ 深圳）有限公司成立于 2021 年，是 TCL 集团孵化的高新技术企业，专注于消费级显示+智能眼镜开发，致力于推动 AI+AR 硬件及场景应用的规模化落地。",
    "length": 86,
    "style": "text-align:center;margin-bottom:8.0pt;line-height:1.1583;font-size:12.0pt;font-family:'STFangsong';font-weight:bold;text-indent:24.0pt",
    "class": "",
    "id": ""
  },
  {
    "tag": "h2",
    "text": "三、AI眼镜为何爆发在即",
    "length": 12,
    "style": "margin-bottom:8.0pt;line-height:1.1583;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_Toc198131829"
  },
  {
    "tag": "h2",
    "text": "五、历史沿革及管理架构",
    "length": 11,
    "style": "margin-bottom:8.0pt;line-height:1.1583;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_Toc198131822"
  },
  {
    "tag": "h2",
    "text": "五、AI眼镜的应用场景",
    "length": 11,
    "style": "margin-bottom:8.0pt;line-height:1.1583;font-size:14.0pt;font-family:'STFangsong';font-weight:bold",
    "class": "",
    "id": "_Toc19

In [13]:
investment = analysis["investment"]["paragraphs"]
center = next(i for i, row in enumerate(investment) if "作为国内AI眼镜领域的链主企业" in row["text"])
window = investment[max(0, center - 3): center + 4]
out = []
for row in window:
    out.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
        "in_headings": row["in_parse_headings"],
    })
print(json.dumps(out, ensure_ascii=False, indent=2))

[
  {
    "index": 38,
    "snippet": "雷鸟创新技术（深圳）有限公司成立于 2021 年，是 TCL 集团孵化的高新技术企业，专注于消费级显示+智能眼镜开发，致力于推动 AI+AR 硬件及场景应用的规模化落地。",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": null,
    "alignment": null,
    "ppr_bold": null,
    "font_size": 12.0,
    "run_bold": {
      "on": 0,
      "off": 0,
      "unset": 2,
      "total": 2,
      "dominant": "unset"
    },
    "html_tag": null,
    "in_headings": false
  },
  {
    "index": 39,
    "snippet": "公司已形成三大核心产品线：具备拍摄与AI交互功能的V3等AI眼镜、搭载全彩MicroLED光波导的AR眼镜X3 Pro，以及轻量化观影类雷鸟 Air 系列。产品应用覆盖实时智能翻译、出行记录、运动监控、智能办公等多元化场景。产品在海内外市场",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": null,
    "alignment": null,
    "ppr_bold": null,
    "font_size": 12.0,
    "run_bold": {
      "on": 0,
      "off": 0,
      "unset": 1,
      "total": 1,
      "dominant": "unset"
    },
    "html_tag": "p",
    "in_headings": false
  },
  {
    "index": 40,
    "snippet": "在技术层面，公司已累计申请专利超400项，其中发明专利占比超过60%，集中布局于MicroLED显示、浮雕光波导、

In [14]:
investment = analysis["investment"]["paragraphs"]
bold_body = []
for row in investment:
    if row["in_parse_headings"]:
        continue
    if row["outlineLvl"] is not None:
        continue
    if row["ppr_bold"] is not True:
        continue
    bold_body.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "alignment": row["alignment"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
    })
print(json.dumps(bold_body[:12], ensure_ascii=False, indent=2))

[
  {
    "index": 1,
    "snippet": "投资报告书",
    "style": [
      "Normal",
      "1"
    ],
    "alignment": "center",
    "font_size": 36.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": "p"
  },
  {
    "index": 13,
    "snippet": "重要声明",
    "style": [
      "Normal",
      "1"
    ],
    "alignment": "center",
    "font_size": 16.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": "p"
  },
  {
    "index": 44,
    "snippet": "AI眼镜行业潜力巨大：",
    "style": [
      "List Paragraph",
      "54"
    ],
    "alignment": null,
    "font_size": 12.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": null
  },
  {
    "index": 47,
    "snippet": "标的是行业龙头民营企业：",
    "style": [
      "List Paragraph",
      "54"
    ],
    "alignment": null,
    "fo

In [15]:
resume = analysis["resume"]["paragraphs"]
block_bold = []
for row in resume:
    style = row.get("html_style", "") or ""
    if "font-weight:bold" not in style:
        continue
    block_bold.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
        "html_style": row["html_style"],
    })
print("block_bold_count", len(block_bold))
print(json.dumps(block_bold[:20], ensure_ascii=False, indent=2))

block_bold_count 7
[
  {
    "index": 0,
    "snippet": "王宇轩",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": null,
    "alignment": "center",
    "ppr_bold": true,
    "font_size": 12.0,
    "run_bold": {
      "on": 2,
      "off": 0,
      "unset": 0,
      "total": 2,
      "dominant": "on"
    },
    "html_tag": "p",
    "html_style": "text-align:center;font-size:12.0pt;font-weight:bold;text-indent:11.95pt;padding-left:-8.9pt"
  },
  {
    "index": 4,
    "snippet": "教育背景",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": null,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 11.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": "p",
    "html_style": "font-size:11.0pt;font-weight:bold"
  },
  {
    "index": 8,
    "snippet": "罗切斯特大学 生物学学士 本科 08/2018-05/2022 美国，罗切斯特",
    "style": [
      "Normal",
      "1"
    ],
    "outlineLvl": null,
    "alignme

In [16]:
resume = analysis["resume"]["paragraphs"]
ppr_bold_rows = [
    {
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "html_tag": row["html_tag"],
        "html_style": row["html_style"],
    }
    for row in resume
    if row["ppr_bold"] is True
]
print("ppr_bold_count", len(ppr_bold_rows))
print(json.dumps(ppr_bold_rows, ensure_ascii=False, indent=2))

ppr_bold_count 12
[
  {
    "index": 0,
    "snippet": "王宇轩",
    "style": [
      "Normal",
      "1"
    ],
    "alignment": "center",
    "ppr_bold": true,
    "font_size": 12.0,
    "run_bold": {
      "on": 2,
      "off": 0,
      "unset": 0,
      "total": 2,
      "dominant": "on"
    },
    "html_tag": "p",
    "html_style": "text-align:center;font-size:12.0pt;font-weight:bold;text-indent:11.95pt;padding-left:-8.9pt"
  },
  {
    "index": 4,
    "snippet": "教育背景",
    "style": [
      "Normal",
      "1"
    ],
    "alignment": null,
    "ppr_bold": true,
    "font_size": 11.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "html_tag": "p",
    "html_style": "font-size:11.0pt;font-weight:bold"
  },
  {
    "index": 5,
    "snippet": "爱丁堡大学 生物信息学 硕士 09/2022-10/2023 英国，爱丁堡",
    "style": [
      "Normal",
      "1"
    ],
    "alignment": "justify",
    "ppr_bold": true,
    "font_size": 10.0,
    "run_bold"

In [17]:
resume = analysis["resume"]["paragraphs"]
rows = [
    {
        "index": row["index"],
        "snippet": row["snippet"],
        "alignment": row["alignment"],
        "font_size": row["ppr_font_size"],
        "run_dominant": row["run_bold"]["dominant"],
        "run_on": row["run_bold"]["on"],
        "run_off": row["run_bold"]["off"],
        "run_unset": row["run_bold"]["unset"],
        "html_bold": "font-weight:bold" in (row["html_style"] or ""),
    }
    for row in resume
    if row["ppr_bold"] is True
]
print(json.dumps(rows, ensure_ascii=False, indent=2))

[
  {
    "index": 0,
    "snippet": "王宇轩",
    "alignment": "center",
    "font_size": 12.0,
    "run_dominant": "on",
    "run_on": 2,
    "run_off": 0,
    "run_unset": 0,
    "html_bold": true
  },
  {
    "index": 4,
    "snippet": "教育背景",
    "alignment": null,
    "font_size": 11.0,
    "run_dominant": "on",
    "run_on": 1,
    "run_off": 0,
    "run_unset": 0,
    "html_bold": true
  },
  {
    "index": 5,
    "snippet": "爱丁堡大学 生物信息学 硕士 09/2022-10/2023 英国，爱丁堡",
    "alignment": "justify",
    "font_size": 10.0,
    "run_dominant": "on",
    "run_on": 5,
    "run_off": 0,
    "run_unset": 3,
    "html_bold": false
  },
  {
    "index": 8,
    "snippet": "罗切斯特大学 生物学学士 本科 08/2018-05/2022 美国，罗切斯特",
    "alignment": "justify",
    "font_size": 10.0,
    "run_dominant": "on",
    "run_on": 5,
    "run_off": 0,
    "run_unset": 1,
    "html_bold": true
  },
  {
    "index": 13,
    "snippet": "金雨茂物投资管理 05/2024-12/2025 中国，南京",
    "alignment": "justify",
    "font_size": 10.0,
    "ru

In [18]:
targets = [
    "爱丁堡大学 生物信息学 硕士",
    "金雨茂物投资管理",
    "爱丁堡大学Dr. Cei 实验室",
    "M.SC. BIOINFORMATICS, University of edinburgh",
    "Jolmo Investment management",
]
blocks = analysis["resume"]["blocks"]
out = {}
for target in targets:
    matches = []
    for block in blocks:
        if target in block["text"]:
            matches.append({
                "tag": block["tag"],
                "text": block["text"][:160],
                "style": block["style"],
                "class": block["class"],
            })
    out[target] = matches[:3]
print(json.dumps(out, ensure_ascii=False, indent=2))

{
  "爱丁堡大学 生物信息学 硕士": [
    {
      "tag": "p",
      "text": "爱丁堡大学 生物信息学 硕士 09/ 20 22-10/2023 英国，爱丁堡",
      "style": "text-align:justify;line-height:1.0;font-size:10.0pt;font-weight:bold",
      "class": ""
    }
  ],
  "金雨茂物投资管理": [
    {
      "tag": "p",
      "text": "金雨茂物投资管理 05/2024- 12/2025 中国， 南京",
      "style": "text-align:justify;font-size:10.0pt;font-weight:bold",
      "class": ""
    }
  ],
  "爱丁堡大学Dr. Cei 实验室": [
    {
      "tag": "p",
      "text": "爱丁堡大学Dr. Cei 实验室 02/202 3 - 09 /202 3 英 国， 爱丁堡",
      "style": "text-align:justify;font-size:10.0pt;font-family:'SimSun';font-weight:bold",
      "class": ""
    }
  ],
  "M.SC. BIOINFORMATICS, University of edinburgh": [],
  "Jolmo Investment management": []
}


In [19]:
resume = analysis["resume"]["paragraphs"]
selected = {5, 13, 21, 27}
out = []
for row in resume:
    if row["index"] not in selected:
        continue
    out.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "chain_has_heading": row["style_chain_has_heading"],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "looks_like_toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "html_style": row["html_style"],
    })
print(json.dumps(out, ensure_ascii=False, indent=2))

[
  {
    "index": 5,
    "snippet": "爱丁堡大学 生物信息学 硕士 09/2022-10/2023 英国，爱丁堡",
    "style": [
      "Normal",
      "1"
    ],
    "chain_has_heading": false,
    "outlineLvl": null,
    "alignment": "justify",
    "ppr_bold": true,
    "font_size": 10.0,
    "run_bold": {
      "on": 5,
      "off": 0,
      "unset": 3,
      "total": 8,
      "dominant": "on"
    },
    "looks_like_toc": false,
    "html_tag": null,
    "html_style": ""
  },
  {
    "index": 13,
    "snippet": "金雨茂物投资管理 05/2024-12/2025 中国，南京",
    "style": [
      "Normal",
      "1"
    ],
    "chain_has_heading": false,
    "outlineLvl": null,
    "alignment": "justify",
    "ppr_bold": true,
    "font_size": 10.0,
    "run_bold": {
      "on": 7,
      "off": 0,
      "unset": 4,
      "total": 11,
      "dominant": "on"
    },
    "looks_like_toc": false,
    "html_tag": null,
    "html_style": ""
  },
  {
    "index": 21,
    "snippet": "爱丁堡大学Dr. Cei 实验室 02/2023-09/2023 英国，爱丁堡",
    "style": [
      "Normal",
   

In [20]:
investment = analysis["investment"]["paragraphs"]
selected = {41, 43, 44, 47, 60, 81}
out = []
for row in investment:
    if row["index"] not in selected:
        continue
    out.append({
        "index": row["index"],
        "snippet": row["snippet"],
        "style": [row["style_name"], row["style_id"]],
        "chain_has_heading": row["style_chain_has_heading"],
        "outlineLvl": row["outlineLvl"],
        "alignment": row["alignment"],
        "ppr_bold": row["ppr_bold"],
        "font_size": row["ppr_font_size"],
        "run_bold": row["run_bold"],
        "looks_like_toc": row["looks_like_toc"],
        "html_tag": row["html_tag"],
        "html_style": row["html_style"],
        "in_headings": row["in_parse_headings"],
    })
print(json.dumps(out, ensure_ascii=False, indent=2))

[
  {
    "index": 41,
    "snippet": "作为国内AI眼镜领域的链主企业，雷鸟创新已完成“核心器件+整机终端+软件内容”三位一体的全链路布局，构建起集设计、制造、算法、内容服务于一体的智能眼镜产业链。面向未来，公司将继续加大在AI融合交互、光学显示、空间感知等前沿技术的投入，力争在",
    "style": [
      "Normal",
      "1"
    ],
    "chain_has_heading": false,
    "outlineLvl": null,
    "alignment": null,
    "ppr_bold": null,
    "font_size": 12.0,
    "run_bold": {
      "on": 0,
      "off": 0,
      "unset": 1,
      "total": 1,
      "dominant": "unset"
    },
    "looks_like_toc": false,
    "html_tag": "p",
    "html_style": "margin-bottom:8.0pt;line-height:1.1583;font-size:12.0pt;font-family:'STFangsong';text-indent:21.0pt",
    "in_headings": false
  },
  {
    "index": 43,
    "snippet": "二、投资亮点",
    "style": [
      "Normal",
      "1"
    ],
    "chain_has_heading": false,
    "outlineLvl": 1,
    "alignment": null,
    "ppr_bold": true,
    "font_size": 14.0,
    "run_bold": {
      "on": 1,
      "off": 0,
      "unset": 0,
      "total": 1,
      "dominant": "on"
    },
    "looks_like_t

In [22]:
resume_doc = Document(str(DOC_RESUME))
structure = {"tables": len(resume_doc.tables), "table_cells_with_images": [], "body_image_paragraphs": []}
for ti, table in enumerate(resume_doc.tables):
    seen = set()
    for ri, row in enumerate(table.rows):
        for ci, cell in enumerate(row.cells):
            key = id(cell._tc)
            if key in seen:
                continue
            seen.add(key)
            for pi, para in enumerate(cell.paragraphs):
                text = clean_text(''.join(run.text for run in para.runs))
                has_drawing = para._element.find('.//' + qn('w:drawing')) is not None
                if has_drawing:
                    structure["table_cells_with_images"].append({
                        "table": ti,
                        "row": ri,
                        "col": ci,
                        "para": pi,
                        "text": text,
                        "text_len": len(text),
                    })
for pi, para in enumerate(resume_doc.paragraphs):
    text = clean_text(''.join(run.text for run in para.runs))
    has_drawing = para._element.find('.//' + qn('w:drawing')) is not None
    if has_drawing:
        structure["body_image_paragraphs"].append({
            "para": pi,
            "text": text,
            "text_len": len(text),
        })
print(json.dumps(structure, ensure_ascii=False, indent=2))

{
  "tables": 2,
  "table_cells_with_images": [
    {
      "table": 0,
      "row": 0,
      "col": 0,
      "para": 0,
      "text": "王宇轩",
      "text_len": 3
    },
    {
      "table": 1,
      "row": 0,
      "col": 0,
      "para": 0,
      "text": "Logan Wang (王宇轩)",
      "text_len": 16
    }
  ],
  "body_image_paragraphs": []
}


In [50]:
resume_parsed = parse_docx(str(DOC_RESUME))
resume_html = resume_parsed.get("html", "")
resume_soup = BeautifulSoup(resume_html, "html.parser")

table_summary = []
for ti, table in enumerate(Document(str(DOC_RESUME)).tables):
    seen = set()
    rows = []
    for ri, row in enumerate(table.rows):
        row_out = []
        for ci, cell in enumerate(row.cells):
            key = id(cell._tc)
            if key in seen:
                row_out.append({"col": ci, "merged": True})
                continue
            seen.add(key)
            row_out.append({
                "col": ci,
                "text": clean_text(cell.text),
                "paragraphs": [clean_text(p.text) for p in cell.paragraphs],
                "has_drawing": any(p._element.find('.//' + qn('w:drawing')) is not None for p in cell.paragraphs),
            })
        rows.append({"row": ri, "cells": row_out})
    table_summary.append({"table": ti, "rows": rows})

img_contexts = []
for idx, img in enumerate(resume_soup.find_all("img"), start=1):
    parent = img.parent
    img_contexts.append({
        "img_index": idx,
        "parent_tag": parent.name if parent else None,
        "parent_text": clean_text(parent.get_text(" ", strip=True)) if parent else "",
        "parent_style": parent.get("style", "") if parent else "",
        "td_style": next((anc.get("style", "") for anc in img.parents if getattr(anc, "name", None) == "td"), ""),
    })

print(json.dumps({
    "table_summary": table_summary,
    "img_contexts": img_contexts,
}, ensure_ascii=False, indent=2))

{
  "table_summary": [
    {
      "table": 0,
      "rows": [
        {
          "row": 0,
          "cells": [
            {
              "col": 0,
              "text": "王宇轩 Loganwon0314@163.com (86) 18913921188 南京市建邺区邺城路27号6-1702",
              "paragraphs": [
                "王宇轩",
                "Loganwon0314@163.com",
                "(86) 18913921188",
                "南京市建邺区邺城路27号6-1702"
              ],
              "has_drawing": true
            },
            {
              "col": 1,
              "merged": true
            }
          ]
        },
        {
          "row": 1,
          "cells": [
            {
              "col": 0,
              "text": "教育背景",
              "paragraphs": [
                "教育背景"
              ],
              "has_drawing": false
            },
            {
              "col": 1,
              "merged": true
            }
          ]
        },
        {
          "row": 2,
          "cells": [
            {
              "col

In [24]:
print(json.dumps({
    "table_count": len(table_summary),
    "image_cell_locations": [
        {
            "table": item["table"],
            "row": item["row"],
            "col": item["col"],
            "text": item["text"],
        }
        for item in structure["table_cells_with_images"]
    ],
    "first_table_first_row": table_summary[0]["rows"][0],
    "second_table_first_row": table_summary[1]["rows"][0],
    "img_contexts": img_contexts[:4],
}, ensure_ascii=False, indent=2))

{
  "table_count": 2,
  "image_cell_locations": [
    {
      "table": 0,
      "row": 0,
      "col": 0,
      "text": "王宇轩"
    },
    {
      "table": 1,
      "row": 0,
      "col": 0,
      "text": "Logan Wang (王宇轩)"
    }
  ],
  "first_table_first_row": {
    "row": 0,
    "cells": [
      {
        "col": 0,
        "text": "王宇轩 Loganwon0314@163.com (86) 18913921188 南京市建邺区邺城路27号6-1702",
        "paragraphs": [
          "王宇轩",
          "Loganwon0314@163.com",
          "(86) 18913921188",
          "南京市建邺区邺城路27号6-1702"
        ],
        "has_drawing": true
      },
      {
        "col": 1,
        "merged": true
      }
    ]
  },
  "second_table_first_row": {
    "row": 0,
    "cells": [
      {
        "col": 0,
        "text": "Logan Wang (王宇轩) Loganwon0314@163.com (86) 18913921188 Room 6-1702, No.27 Yecheng Road, Jianye District, Nanjing 210019, China",
        "paragraphs": [
          "Logan Wang (王宇轩)",
          "Loganwon0314@163.com",
          "(86) 18913921188",
  

In [48]:
probe = {
    "img_contexts": img_contexts[:6],
    "image_cell_locations": [
        {
            "table": item["table"],
            "row": item["row"],
            "col": item["col"],
            "text": item["text"],
        }
        for item in structure["table_cells_with_images"]
    ],
    "first_table_first_row": table_summary[0]["rows"][0] if table_summary else None,
}
print(json.dumps(probe, ensure_ascii=False, indent=2))

{
  "img_contexts": [
    {
      "img_index": 1,
      "parent_tag": "td",
      "parent_text": "王宇轩 Loganwon0314@163.com (86) 1 8913921188 南京市建邺区邺城路27号6-1702",
      "parent_style": "width:100.0%;vertical-align:middle;border-top:none;border-bottom:none;border-left:none;border-right:none;padding:0 5.4pt",
      "td_style": "width:100.0%;vertical-align:middle;border-top:none;border-bottom:none;border-left:none;border-right:none;padding:0 5.4pt"
    }
  ],
  "image_cell_locations": [
    {
      "table": 0,
      "row": 0,
      "col": 0,
      "text": "王宇轩"
    },
    {
      "table": 1,
      "row": 0,
      "col": 0,
      "text": "Logan Wang (王宇轩)"
    }
  ],
  "first_table_first_row": {
    "row": 0,
    "cells": [
      {
        "col": 0,
        "text": "王宇轩 Loganwon0314@163.com (86) 18913921188 南京市建邺区邺城路27号6-1702",
        "paragraphs": [
          "王宇轩",
          "Loganwon0314@163.com",
          "(86) 18913921188",
          "南京市建邺区邺城路27号6-1702"
        ],
        "has_drawi

In [55]:
import importlib
from app.core.file import file_parser as file_parser_mod
file_parser_mod = importlib.reload(file_parser_mod)
fresh_resume_html = file_parser_mod.parse_docx(str(DOC_RESUME)).get("html", "")
fresh_resume_soup = BeautifulSoup(fresh_resume_html, "html.parser")
img = fresh_resume_soup.find("img")
parent_td = img.find_parent("td") if img else None
child_tags = []
if parent_td:
    for child in parent_td.children:
        name = getattr(child, "name", None)
        if name:
            child_tags.append(name)
img_info = None
if img:
    img_info = {
        "style": img.get("style", ""),
        "data_koto_layout": img.get("data-koto-layout"),
        "display": img.get("display"),
    }
print(json.dumps({
    "img_info": img_info,
    "child_tags": child_tags,
}, ensure_ascii=False, indent=2))

{
  "img_info": {
    "style": "display:inline-block;vertical-align:middle;margin:4px 8px;width:74px;max-width:100%;",
    "data_koto_layout": "top-bottom",
    "display": null
  },
  "child_tags": [
    "p",
    "p",
    "p",
    "p",
    "p"
  ]
}


In [54]:
import importlib
from app.core.file import file_parser as file_parser_mod
file_parser_mod = importlib.reload(file_parser_mod)
sample_html = '<table><tr><td><p>email<img style="display:inline-block;width:74px;max-width:100%;" src="x" /></p><p>phone</p></td></tr></table>'
print(file_parser_mod._extract_images_from_paragraphs(sample_html))

<table><tr><td><p>email</p><p class="koto-docx-image-row"><img data-koto-layout="top-bottom" src="x" style="display:inline-block;width:74px;max-width:100%;"/></p><p>phone</p></td></tr></table>


In [57]:
import os
from pathlib import Path
import pytest
repo_root = Path.cwd().resolve()
if repo_root.name == ".koto":
    repo_root = repo_root.parent
os.chdir(repo_root)
pytest_args = [
    "tests/test_docx_rendering.py::TestImages::test_table_cell_inline_images_become_dedicated_image_rows",
    "tests/test_docx_rendering.py::TestTypography::test_outline_level_body_sentence_is_not_promoted_to_heading",
    "tests/test_docx_rendering.py::TestTypography::test_generic_chinese_title_style_is_not_treated_as_structural_heading",
    "-q",
]
result = pytest.main(pytest_args)
print({"pytest_exit_code": result})
assert result == 0


tests/test_docx_rendering.py::TestImages::test_table_cell_inline_images_become_dedicated_image_rows PASSED [ 33%]
tests/test_docx_rendering.py::TestTypography::test_outline_level_body_sentence_is_not_promoted_to_heading PASSED [ 66%]
tests/test_docx_rendering.py::TestTypography::test_generic_chinese_title_style_is_not_treated_as_structural_heading PASSED [100%]

: 